# N=8 Kuramoto benchmark — polynomial discovery of a sinusoidal interaction law

This notebook mirrors the canonical N=8 TIDES benchmark but changes the data-generating edge law to

\[
\dot x_i=\sum_j W_{ij}^{(r)}\sin(x_j-x_i).
\]

The inference side is **not** given a sine atom.  Instead it receives a translation-invariant polynomial library in the relative coordinate

\[
z_{ij}=\frac{x_j-x_i}{s},
\]

with powers \(z,z^2,\ldots,z^p\).  The final cells ask whether the recovered shared polynomial law can be compressed back to \(\sin(x_j-x_i)\).

Three diagnostics are kept separate:

1. **blind Step 2 → Step 3 → Step 4**, to see what the current pipeline does;
2. **oracle change-support Step 3 → Step 4**, used only to isolate the dynamics-identification question from support-selection errors.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import importlib

import tides
import pairwise_local_interaction_library as plil

print("TIDES loaded from:")
print(tides.__file__)
print("\nPairwise local-interaction library loaded from:")
print(plil.__file__)


TIDES loaded from:
/home/liu.xuanc/ondemand/TIDES/tides.py

Pairwise local-interaction library loaded from:
/home/liu.xuanc/ondemand/TIDES/pairwise_local_interaction_library.py


In [2]:
# ============================================================
# Cell 2 — Kuramoto N=8 benchmark truth
# ============================================================
from itertools import combinations

SEED = 20260811
N = 8
DT = 5.0e-4
STAGE_DURATION = 0.060
N_STAGES = 6
INTERVALS_PER_STAGE = int(round(STAGE_DURATION / DT))
assert INTERVALS_PER_STAGE == 120

CANDIDATE_EDGES = tuple(combinations(range(1, N + 1), 2))
EDGE_INDEX_1B = {edge: m for m, edge in enumerate(CANDIDATE_EDGES)}
assert len(CANDIDATE_EDGES) == 28

weight_rng = np.random.default_rng(SEED)
EDGE_WEIGHT = {
    edge: int(weight_rng.integers(800, 1201)) / 1000.0
    for edge in CANDIDATE_EDGES
}

# Same temporal topology schedule as the canonical benchmark.
SNAPSHOTS = (
    ((1, 3), (2, 5), (2, 8), (3, 5), (3, 8),
     (4, 6), (4, 7), (5, 6), (6, 8), (7, 8)),
    ((1, 3), (1, 5), (2, 3), (2, 8), (3, 5),
     (3, 8), (4, 6), (4, 7), (5, 6), (7, 8)),
    ((1, 3), (1, 6), (2, 3), (2, 7), (2, 8),
     (3, 8), (4, 6), (4, 7), (5, 6), (7, 8)),
    ((1, 3), (1, 4), (1, 6), (2, 7), (2, 8),
     (3, 8), (4, 5), (4, 6), (5, 6), (7, 8)),
    ((1, 3), (1, 6), (2, 7), (2, 8), (3, 8),
     (4, 5), (5, 6), (5, 7), (6, 7), (7, 8)),
    ((1, 3), (1, 8), (2, 8), (3, 8), (4, 5),
     (4, 8), (5, 6), (5, 7), (6, 7), (7, 8)),
)

X0 = np.array([
    -0.319989, -0.301418, 0.354093, -0.213312,
    -0.026615,  0.067347, 0.282008,  0.157887,
], dtype=float)
X0 -= X0.mean()


def edge_field(x, edge):
    i, j = edge
    i -= 1
    j -= 1
    q = EDGE_WEIGHT[edge] * np.sin(x[j] - x[i])
    out = np.zeros_like(x, dtype=float)
    out[i] += q
    out[j] -= q
    return out


def stage_field(x, stage):
    out = np.zeros(N, dtype=float)
    for edge in SNAPSHOTS[stage]:
        out += edge_field(x, edge)
    return out


def rk4_step(x, dt, stage):
    k1 = stage_field(x, stage)
    k2 = stage_field(x + 0.5 * dt * k1, stage)
    k3 = stage_field(x + 0.5 * dt * k2, stage)
    k4 = stage_field(x + dt * k3, stage)
    return x + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


def is_connected(edge_set):
    adjacency = {i: set() for i in range(1, N + 1)}
    for i, j in edge_set:
        adjacency[i].add(j)
        adjacency[j].add(i)
    seen, stack = {1}, [1]
    while stack:
        u = stack.pop()
        for v in adjacency[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return len(seen) == N

transition_supports_true = []
for k in range(N_STAGES - 1):
    before = set(SNAPSHOTS[k])
    after = set(SNAPSHOTS[k + 1])
    transition_supports_true.append(tuple(sorted(before ^ after)))

assert all(len(snapshot) == 10 for snapshot in SNAPSHOTS)
assert all(is_connected(snapshot) for snapshot in SNAPSHOTS)
assert all(len(S) == 4 for S in transition_supports_true)

true_change_times = np.arange(1, N_STAGES) * STAGE_DURATION
true_transition_indices = np.rint(true_change_times / DT).astype(int)

print("=" * 90)
print("TIDES N=8 Kuramoto benchmark")
print("=" * 90)
print("candidate edges M       :", len(CANDIDATE_EDGES))
print("active edges / stage    :", [len(G) for G in SNAPSHOTS])
print("true edge law           : sin(x_neighbor - x_self)")
print("true change times       :", true_change_times)
print("true transition indices :", true_transition_indices)
for k, S in enumerate(transition_supports_true, 1):
    print(f"transition {k}: {S}")


TIDES N=8 Kuramoto benchmark
candidate edges M       : 28
active edges / stage    : [10, 10, 10, 10, 10, 10]
true edge law           : sin(x_neighbor - x_self)
true change times       : [0.06 0.12 0.18 0.24 0.3 ]
true transition indices : [120 240 360 480 600]
transition 1: ((1, 5), (2, 3), (2, 5), (6, 8))
transition 2: ((1, 5), (1, 6), (2, 7), (3, 5))
transition 3: ((1, 4), (2, 3), (4, 5), (4, 7))
transition 4: ((1, 4), (4, 6), (5, 7), (6, 7))
transition 5: ((1, 6), (1, 8), (2, 7), (4, 8))


In [3]:
# ============================================================
# Cell 3 — Generate Kuramoto trajectory
# ============================================================
n_total_intervals = N_STAGES * INTERVALS_PER_STAGE
T = np.arange(n_total_intervals + 1, dtype=float) * DT
X = np.empty((n_total_intervals + 1, N), dtype=float)
X[0] = X0

true_stage_of_interval = np.repeat(
    np.arange(N_STAGES, dtype=int), INTERVALS_PER_STAGE
)
for n, stage in enumerate(true_stage_of_interval):
    X[n + 1] = rk4_step(X[n], DT, int(stage))

t = T.copy()
state_displacement = np.linalg.norm(X[-1] - X[0])
max_abs_state = np.max(np.abs(X))
net_sum_change = float(X[-1].sum() - X[0].sum())

print("=" * 90)
print("Kuramoto trajectory generated")
print("=" * 90)
print("X shape                :", X.shape)
print("||X(T)-X(0)||          :", f"{state_displacement:.6e}")
print("max |x_i(t)|           :", f"{max_abs_state:.6e}")
print("sum_i x_i change       :", f"{net_sum_change:.6e}")
print("all finite             :", np.isfinite(X).all())

assert X.shape == (721, 8)
assert np.allclose(np.diff(t), DT)
assert np.isfinite(X).all()
# Pairwise Kuramoto coupling is antisymmetric and conserves the node-state sum.
assert abs(net_sum_change) < 1e-10


Kuramoto trajectory generated
X shape                : (721, 8)
||X(T)-X(0)||          : 4.527580e-01
max |x_i(t)|           : 3.540929e-01
sum_i x_i change       : -3.608225e-16
all finite             : True


In [4]:
# ============================================================
# Cell 4 — Formal TIDES Step 1: change-point detection
# ============================================================

# A. Controlled-count regression
s1_controlled = tides.detect_changes(
    X,
    t,
    method="secant",
    n_changes=5,
    min_separation=10,
)

print("=" * 90)
print("TIDES Step 1 — controlled-count regression")
print("=" * 90)
print("true indices       :", true_transition_indices)
print("detected indices   :", s1_controlled.transition_indices)
print("index errors       :", s1_controlled.transition_indices - true_transition_indices)
print("true times         :", true_change_times)
print("detected times     :", s1_controlled.transition_times)

# B. Fully blind robust threshold
V_SECANT = np.diff(X, axis=0) / DT
jump = np.linalg.norm(V_SECANT[1:] - V_SECANT[:-1], axis=1)
jump_median = np.median(jump)
jump_mad = np.median(np.abs(jump - jump_median))
blind_threshold = jump_median + 20.0 * jump_mad

s1_blind = tides.detect_changes(
    X,
    t,
    method="secant",
    threshold=blind_threshold,
    min_separation=1,
)

print("\n" + "=" * 90)
print("TIDES Step 1 — fully blind detection")
print("=" * 90)
print("jump median        :", f"{jump_median:.6e}")
print("jump MAD           :", f"{jump_mad:.6e}")
print("blind threshold    :", f"{blind_threshold:.6e}")
print("detected count     :", s1_blind.n_transitions)
print("detected indices   :", s1_blind.transition_indices)
print("detected times     :", s1_blind.transition_times)

controlled_exact = np.array_equal(
    s1_controlled.transition_indices,
    true_transition_indices,
)
blind_exact = np.array_equal(
    s1_blind.transition_indices,
    true_transition_indices,
)

print("\n" + "=" * 90)
print("controlled exact   :", controlled_exact)
print("blind exact        :", blind_exact)
assert controlled_exact
assert blind_exact
print("TIDES STEP 1: PASS")
print("=" * 90)


TIDES Step 1 — controlled-count regression
true indices       : [120 240 360 480 600]
detected indices   : [120 240 360 480 600]
index errors       : [0 0 0 0 0]
true times         : [0.06 0.12 0.18 0.24 0.3 ]
detected times     : [0.06 0.12 0.18 0.24 0.3 ]

TIDES Step 1 — fully blind detection
jump median        : 2.052780e-03
jump MAD           : 3.333185e-04
blind threshold    : 8.719151e-03
detected count     : 5
detected indices   : [120 240 360 480 600]
detected times     : [0.06 0.12 0.18 0.24 0.3 ]

controlled exact   : True
blind exact        : True
TIDES STEP 1: PASS


In [5]:
# ============================================================
# Cell 5 — PREPROCESSING
#          Step-1 segmentation + 4-point midpoint reconstruction
# ============================================================
# From this point onward segmentation is determined only by
# the BLIND Step-1 output.


detected_transition_indices = s1_blind.transition_indices.copy()
detected_transition_times = s1_blind.transition_times.copy()

V_SECANT = np.diff(X, axis=0) / DT
n_intervals = len(V_SECANT)

segment_bounds = np.concatenate(
    ([0], detected_transition_indices, [n_intervals])
)
segment_slices = tuple(
    slice(int(segment_bounds[k]), int(segment_bounds[k + 1]))
    for k in range(len(segment_bounds) - 1)
)
segment_interval_counts = np.array(
    [sl.stop - sl.start for sl in segment_slices],
    dtype=int,
)

stage_of_interval = np.empty(n_intervals, dtype=int)
for stage, sl in enumerate(segment_slices):
    stage_of_interval[sl] = stage

obs_interval_indices = []
X_MID = []
V_MID = []
OBS_STAGE = []
T_MID = []

for n in range(1, n_intervals - 1):
    same_stage = (
        stage_of_interval[n - 1]
        == stage_of_interval[n]
        == stage_of_interval[n + 1]
    )
    if not same_stage:
        continue

    x_mid = (
        -X[n - 1]
        + 9.0 * X[n]
        + 9.0 * X[n + 1]
        - X[n + 2]
    ) / 16.0

    v_mid = (
        X[n - 1]
        - 27.0 * X[n]
        + 27.0 * X[n + 1]
        - X[n + 2]
    ) / (24.0 * DT)

    obs_interval_indices.append(n)
    X_MID.append(x_mid)
    V_MID.append(v_mid)
    T_MID.append((t[n] + t[n + 1]) / 2.0)
    OBS_STAGE.append(stage_of_interval[n])

obs_interval_indices = np.asarray(obs_interval_indices, dtype=int)
X_MID = np.asarray(X_MID, dtype=float)
V_MID = np.asarray(V_MID, dtype=float)
T_MID = np.asarray(T_MID, dtype=float)
OBS_STAGE = np.asarray(OBS_STAGE, dtype=int)

TIDES_OBS = {
    "t_state": t.copy(),
    "x_state": X.copy(),
    "detected_transition_indices": detected_transition_indices.copy(),
    "detected_transition_times": detected_transition_times.copy(),
    "segment_bounds": segment_bounds.copy(),
    "segment_interval_counts": segment_interval_counts.copy(),
    "obs_interval_indices": obs_interval_indices.copy(),
    "t_mid": T_MID.copy(),
    "x_mid": X_MID.copy(),
    "velocity_mid": V_MID.copy(),
    "stage_of_observation": OBS_STAGE.copy(),
}

samples_per_stage = np.bincount(OBS_STAGE, minlength=N_STAGES)

print("=" * 90)
print("Cell 5 — PREPROCESSING")
print("=" * 90)
print("detected transitions   :", detected_transition_indices)
print("segment lengths        :", segment_interval_counts.tolist())
print("inference observations :", len(X_MID))
print("samples / stage        :", samples_per_stage.tolist())
print("X_MID shape            :", X_MID.shape)
print("V_MID shape            :", V_MID.shape)
print("OBS_STAGE shape        :", OBS_STAGE.shape)

assert X_MID.shape == (708, 8)
assert V_MID.shape == (708, 8)
assert OBS_STAGE.shape == (708,)
assert np.array_equal(samples_per_stage, np.array([118] * 6))


Cell 5 — PREPROCESSING
detected transitions   : [120 240 360 480 600]
segment lengths        : [120, 120, 120, 120, 120, 120]
inference observations : 708
samples / stage        : [118, 118, 118, 118, 118, 118]
X_MID shape            : (708, 8)
V_MID shape            : (708, 8)
OBS_STAGE shape        : (708,)


In [6]:
# ============================================================
# Cell 6 — PREPROCESSING DIAGNOSTIC
#          4-point vs 6-point numerical-resolution audit
# ============================================================

import math


def finite_difference_weights(nodes, derivative_order):
    nodes = np.asarray(nodes, dtype=float)
    n = len(nodes)
    A = np.vstack([nodes**k for k in range(n)])
    b = np.zeros(n, dtype=float)
    b[derivative_order] = math.factorial(derivative_order)
    return np.linalg.solve(A, b)


z4 = np.array([-1.5, -0.5, 0.5, 1.5], dtype=float)
z6 = np.array([-2.5, -1.5, -0.5, 0.5, 1.5, 2.5], dtype=float)

w4_x = finite_difference_weights(z4, 0)
w4_d = finite_difference_weights(z4, 1)
w6_x = finite_difference_weights(z6, 0)
w6_d = finite_difference_weights(z6, 1)

rel_state_46 = []
rel_velocity_46 = []

for n in range(2, n_intervals - 2):
    stencil_stages = {
        stage_of_interval[n + j]
        for j in (-2, -1, 0, 1, 2)
    }
    if len(stencil_stages) != 1:
        continue

    p4 = X[n - 1 : n + 3]
    p6 = X[n - 2 : n + 4]

    x4 = (w4_x[:, None] * p4).sum(axis=0)
    x6 = (w6_x[:, None] * p6).sum(axis=0)
    v4 = (w4_d[:, None] * p4).sum(axis=0) / DT
    v6 = (w6_d[:, None] * p6).sum(axis=0) / DT

    rel_state_46.append(
        np.linalg.norm(x6 - x4) / max(np.linalg.norm(x6), 1e-15)
    )
    rel_velocity_46.append(
        np.linalg.norm(v6 - v4) / max(np.linalg.norm(v6), 1e-15)
    )

rel_state_46 = np.asarray(rel_state_46)
rel_velocity_46 = np.asarray(rel_velocity_46)

resolution_rel_max = max(
    float(rel_state_46.max()),
    float(rel_velocity_46.max()),
    1e-14,
)

# Conservative externally calibrated uncertainty floor.
UNCERTAINTY_FLOOR = 20.0 * resolution_rel_max

TIDES_OBS.update({
    "rel_state_4v6": rel_state_46.copy(),
    "rel_velocity_4v6": rel_velocity_46.copy(),
    "resolution_rel_max": resolution_rel_max,
    "uncertainty_floor": UNCERTAINTY_FLOOR,
})

print("=" * 90)
print("Cell 6 — PREPROCESSING DIAGNOSTIC")
print("=" * 90)
print("resolution samples       :", len(rel_state_46))
print("max relative state 4-vs-6:", f"{rel_state_46.max():.6e}")
print("max relative vel. 4-vs-6 :", f"{rel_velocity_46.max():.6e}")
print("median relative velocity :", f"{np.median(rel_velocity_46):.6e}")
print("resolution relative max  :", f"{resolution_rel_max:.6e}")
print("uncertainty floor        :", f"{UNCERTAINTY_FLOOR:.6e}")

assert np.isfinite(UNCERTAINTY_FLOOR)
assert UNCERTAINTY_FLOOR > 0.0


Cell 6 — PREPROCESSING DIAGNOSTIC
resolution samples       : 696
max relative state 4-vs-6: 5.191482e-13
max relative vel. 4-vs-6 : 4.025360e-13
median relative velocity : 2.581809e-13
resolution relative max  : 5.191482e-13
uncertainty floor        : 1.038296e-11


In [7]:
# ============================================================
# Cell 7 — BLIND relative-coordinate polynomial library
# ============================================================
# We give inference only the physically natural translation-invariant
# coordinate d = x_neighbor - x_self.  No sine atom is supplied.
# The polynomial library contains every power z^p, p=1,...,POLY_DEGREE,
# where z=d/DIFF_SCALE.  Even powers are intentionally retained; if the
# recovered law is odd, that must emerge from the data.
# ============================================================
import importlib
import math
import pairwise_local_interaction_library
importlib.reload(pairwise_local_interaction_library)

from pairwise_local_interaction_library import (
    PairwiseLocalInteractionAtom,
    build_candidate_pairwise_local_interaction_library,
)

M = len(CANDIDATE_EDGES)
K = len(detected_transition_indices)
POLY_DEGREE = 11

D = np.zeros((N, M), dtype=float)
for m, (i, j) in enumerate(CANDIDATE_EDGES):
    D[i - 1, m] = -1.0
    D[j - 1, m] = +1.0

# Max-absolute scaling keeps |z|<=1 on the sampled candidate-pair domain,
# which is useful for a high-order polynomial basis.
tail = np.array([i - 1 for i, _ in CANDIDATE_EDGES], dtype=int)
head = np.array([j - 1 for _, j in CANDIDATE_EDGES], dtype=int)
D_PHYSICAL = X_MID[:, head] - X_MID[:, tail]
DIFF_SCALE = float(np.max(np.abs(D_PHYSICAL)))
if DIFF_SCALE <= 1e-14:
    DIFF_SCALE = 1.0


def make_relative_power_atom(power, scale):
    p = int(power)
    s = float(scale)
    def evaluate(xs, xr):
        z = (xr - xs) / s
        return np.stack([z**p, (-z)**p], axis=-1)
    return PairwiseLocalInteractionAtom(
        name=f"rel_d^{p}",
        evaluate=evaluate,
        description=(
            f"Relative-coordinate polynomial atom: z^{{{p}}}, "
            "z=(x_head-x_tail)/scale; head receives (-z)^p."
        ),
        swap_equivariant=True,
        metadata={"power": p, "difference_scale": s},
    )

RELATIVE_POLY_ATOMS = tuple(
    make_relative_power_atom(p, DIFF_SCALE)
    for p in range(1, POLY_DEGREE + 1)
)

LOCAL_LIBRARY = build_candidate_pairwise_local_interaction_library(
    X_MID,
    D,
    RELATIVE_POLY_ATOMS,
    check_sampled_swap_equivariance=True,
)
EDGE_PSI = LOCAL_LIBRARY.endpoint_features
L = LOCAL_LIBRARY.n_features

# Benchmark-only reference: Taylor projection of sin(d) into the same basis.
# Step 4 fixes theta[0]=1, so the corresponding topology amplitudes are
# W_hat = DIFF_SCALE * W_true and the gauge-normalized law is
#   theta(z) = sin(DIFF_SCALE*z) / DIFF_SCALE.
THETA_TAYLOR_GAUGE = np.zeros(L, dtype=float)
for p in range(1, POLY_DEGREE + 1):
    if p % 2 == 1:
        k = (p - 1) // 2
        THETA_TAYLOR_GAUGE[p - 1] = (
            ((-1.0)**k) * DIFF_SCALE**(p - 1) / math.factorial(p)
        )

# Numerical approximation audit over the sampled candidate-pair domain.
z_all = (D_PHYSICAL / DIFF_SCALE).ravel()
d_all = D_PHYSICAL.ravel()
f_taylor_phys = DIFF_SCALE * sum(
    THETA_TAYLOR_GAUGE[p - 1] * z_all**p
    for p in range(1, POLY_DEGREE + 1)
)
f_true = np.sin(d_all)
taylor_rel_error = np.linalg.norm(f_taylor_phys - f_true) / np.linalg.norm(f_true)
taylor_max_error = np.max(np.abs(f_taylor_phys - f_true))

print("=" * 90)
print("RELATIVE-COORDINATE POLYNOMIAL LIBRARY")
print("=" * 90)
print("polynomial degree       :", POLY_DEGREE)
print("features per edge L     :", L)
print("difference scale s      :", f"{DIFF_SCALE:.12e}")
print("sampled d range         :", (float(d_all.min()), float(d_all.max())))
print("feature labels          :", LOCAL_LIBRARY.feature_labels)
print("endpoint feature shape  :", EDGE_PSI.shape)
print("Taylor gauge theta      :", np.array2string(THETA_TAYLOR_GAUGE, precision=12))
print("degree-p Taylor rel.err :", f"{taylor_rel_error:.12e}")
print("degree-p Taylor max.err :", f"{taylor_max_error:.12e}")
print("uncertainty floor       :", f"{UNCERTAINTY_FLOOR:.12e}")

assert D.shape == (N, M)
assert L == POLY_DEGREE
assert EDGE_PSI.shape == (X_MID.shape[0], M, L, 2)
assert np.isclose(THETA_TAYLOR_GAUGE[0], 1.0)
print("\nNo sinusoidal candidate was supplied to inference.")


RELATIVE-COORDINATE POLYNOMIAL LIBRARY
polynomial degree       : 11
features per edge L     : 11
difference scale s      : 6.728386482590e-01
sampled d range         : (-0.566033507536893, 0.6728386482589637)
feature labels          : ('rel_d^1', 'rel_d^2', 'rel_d^3', 'rel_d^4', 'rel_d^5', 'rel_d^6', 'rel_d^7', 'rel_d^8', 'rel_d^9', 'rel_d^10', 'rel_d^11')
endpoint feature shape  : (708, 28, 11, 2)
Taylor gauge theta      : [ 1.000000000000e+00  0.000000000000e+00 -7.545197443182e-02
  0.000000000000e+00  1.707900133698e-03  0.000000000000e+00
 -1.840920531713e-05  0.000000000000e+00  1.157509074082e-07
  0.000000000000e+00 -4.763800639758e-10]
degree-p Taylor rel.err : 1.965216564446e-13
degree-p Taylor max.err : 9.281464485866e-13
uncertainty floor       : 1.038296486606e-11

No sinusoidal candidate was supplied to inference.


In [8]:
# ============================================================
# Cell 8 — TIDES STEP 2: dense vs scalable (exploratory)
# ============================================================
# IMPORTANT: for this benchmark Step 2 is no longer required to be an
# exact-support oracle.  We report TP/FP/FN but do not assert exact recovery.
# ============================================================
import importlib
import step2_change_structure
importlib.reload(step2_change_structure)
from step2_change_structure import infer_change_structure_from_observations

DETECTED_TRANSITION_INDICES = np.asarray(s1_blind.transition_indices, dtype=int)
DETECTED_TRANSITION_TIMES = t[DETECTED_TRANSITION_INDICES]

COMMON_STEP2 = dict(
    Y=V_MID,
    D=D,
    edge_features=EDGE_PSI,
    stage_of_sample=OBS_STAGE,
    hypothesis="varying_structure",
    transition_indices=DETECTED_TRANSITION_INDICES,
    transition_times=DETECTED_TRANSITION_TIMES,
    edge_labels=CANDIDATE_EDGES,
    uncertainty_floor=UNCERTAINTY_FLOOR,
    solver_method="group_bpdn_prefix",
    refit_method="dense_lstsq",
    require_solver_convergence=True,
    require_floor_reached=True,
    return_design=False,
)

print("=" * 90)
print("STEP 2 — DENSE REFERENCE")
print("=" * 90)
STEP2_DENSE = infer_change_structure_from_observations(
    **COMMON_STEP2,
    backend="dense",
    solver_kwargs={
        "max_iter": 10000,
        "tol": 1e-8,
        "check_every": 50,
        "verbose": True,
    },
)

print("\n" + "=" * 90)
print("STEP 2 — SCALABLE KKT-WORKING-SET")
print("=" * 90)
STEP2_SCALABLE = infer_change_structure_from_observations(
    **COMMON_STEP2,
    backend="scalable",
    scalable_solver="working_set",
    solver_kwargs={
        "max_iter": 10000,
        "tol": 1e-8,
        "check_every": 50,
        "seed_batch_size": 8,
        "seed_growth_factor": 1.0,
        "kkt_tol": 1e-7,
        "verbose": True,
    },
)

dense_set = set(STEP2_DENSE.selected_groups)
scalable_set = set(STEP2_SCALABLE.selected_groups)
same_support = dense_set == scalable_set

TRUE_CHANGE_GROUPS = {
    (k, EDGE_INDEX_1B[edge])
    for k, support in enumerate(transition_supports_true)
    for edge in support
}
TP = len(scalable_set & TRUE_CHANGE_GROUPS)
FP = len(scalable_set - TRUE_CHANGE_GROUPS)
FN = len(TRUE_CHANGE_GROUPS - scalable_set)
support_exact = scalable_set == TRUE_CHANGE_GROUPS
recall = TP / len(TRUE_CHANGE_GROUPS)

print("\n" + "=" * 90)
print("STEP-2 RESULT")
print("=" * 90)
print("dense selected groups       :", len(dense_set))
print("scalable selected groups    :", len(scalable_set))
print("same support                :", same_support)
print("relative residual           :", f"{STEP2_SCALABLE.relative_residual:.12e}")
print("uncertainty floor           :", f"{UNCERTAINTY_FLOOR:.12e}")
print("true groups                 :", len(TRUE_CHANGE_GROUPS))
print("TP / FP / FN                :", TP, "/", FP, "/", FN)
print("recall                      :", f"{recall:.3f}")
print("support exact               :", support_exact)

for c in STEP2_SCALABLE.constraints:
    print(
        f"transition {c.transition_ordinal + 1} @ {c.transition_time:.6f}: "
        f"{c.support_labels}"
    )

assert STEP2_DENSE.relative_residual <= UNCERTAINTY_FLOOR
assert STEP2_SCALABLE.relative_residual <= UNCERTAINTY_FLOOR
STEP2_RESULT = STEP2_SCALABLE


STEP 2 — DENSE REFERENCE
Starting grouped basis-pursuit denoising...
  samples=5664 | coefficients=1540 | groups=140
  residual radius=1.178e-11 | relative target=1.038e-11
  backend=dense-svd Douglas-Rachford | rank=137/1540 | DR step=5.324e-02
  support selection=disabled here; solver returns group ranking only
  [iter      1] fixed-point=7.748e-01 | rel-res=1.038e-11
  [iter    500] fixed-point=5.328e-09 | rel-res=1.036e-11
Grouped basis-pursuit denoising complete.
  stop reason=Douglas-Rachford fixed-point tolerance reached
  iterations=500 | fixed-point=5.328e-09 | feasible=True
  relative residual=1.036e-11 | objective=1.155269e+01
  dual max-ratio=1.000000e+00 | duality gap=1.127e+01
Starting ranked-prefix support certification...
  candidate groups=140 | prefix cap=140 | relative uncertainty floor=1.038e-11
  [prefix    1] rel-res=9.708e-01 | added=(4, 4)
  [prefix    2] rel-res=8.275e-01 | added=(3, 25)
  [prefix    3] rel-res=7.223e-01 | added=(4, 21)
  [prefix    4] rel-res=

In [9]:
# ============================================================
# Cell 9 — Blind selected-support Step 3
# ============================================================
import importlib
import solvers_linear_regression
import step3_vector_field
importlib.reload(solvers_linear_regression)
importlib.reload(step3_vector_field)
from step3_vector_field import reconstruct_vector_field_from_observations

STEP3_BLIND = reconstruct_vector_field_from_observations(
    Y=V_MID,
    D=D,
    edge_features=EDGE_PSI,
    stage_of_sample=OBS_STAGE,
    change_constraints_or_supports=STEP2_RESULT,
    solver_method="dense_lstsq",
    solver_kwargs={
        "rcond": None,
        "compute_raw_svd_diagnostics": True,
        "verbose": True,
    },
    design_mode="dense",
    return_design=True,
)

print("=" * 90)
print("BLIND STEP 3")
print("=" * 90)
print("B_stages shape          :", STEP3_BLIND.B_stages.shape)
print("parameters              :", STEP3_BLIND.parameter_count)
print("rank                    :", f"{STEP3_BLIND.rank}/{STEP3_BLIND.parameter_count}")
print("identifiable            :", STEP3_BLIND.identifiable)
print("relative residual       :", f"{STEP3_BLIND.relative_residual:.12e}")
print("uncertainty floor       :", f"{UNCERTAINTY_FLOOR:.12e}")
assert STEP3_BLIND.relative_residual <= UNCERTAINTY_FLOOR


Starting linear least-squares solve...
  samples=5664 | features=660 | outputs=1 | method=dense_lstsq
Linear least-squares solve complete.
  relative residual=5.070e-13 | normal-eq residual=1.159e-03
  rank=418/660 | cond(scaled)=inf | cond(raw)=inf
BLIND STEP 3
B_stages shape          : (6, 28, 11)
parameters              : 660
rank                    : 418/660
identifiable            : False
relative residual       : 5.070459077083e-13
uncertainty floor       : 1.038296486606e-11


In [10]:
# ============================================================
# Cell 10 — Current Step 4 on the blind Step-3 representative
# ============================================================
import importlib
import step4_source_decomposition
importlib.reload(step4_source_decomposition)
from step4_source_decomposition import decompose_vector_field_sources

STEP4_BLIND = decompose_vector_field_sources(
    STEP3_BLIND,
    hypothesis="shared_interaction_law",
    normalization="unit_norm",
    reference_component=0,
    component_labels=LOCAL_LIBRARY.component_labels,
)

print("=" * 90)
print("CURRENT STEP 4 — BLIND REPRESENTATIVE")
print("=" * 90)
print("theta                  :", np.array2string(STEP4_BLIND.theta, precision=12))
print("rank-1 energy fraction :", f"{STEP4_BLIND.rank1_energy_fraction:.12e}")
print("s2/s1                  :", f"{STEP4_BLIND.second_to_first_singular_ratio:.12e}")
print("rank-1 residual        :", f"{STEP4_BLIND.relative_residual:.12e}")


CURRENT STEP 4 — BLIND REPRESENTATIVE
theta                  : [-1.102291191389e-09 -6.302838018168e-08 -5.858328522082e-08
 -1.850841536261e-07  2.144005295384e-05 -3.827125552137e-04
  2.481142931123e-03 -6.078988507493e-03 -1.659261531667e-02
 -1.287110960568e-01  9.915214983896e-01]
rank-1 energy fraction : 9.419997095436e-01
s2/s1                  : 2.379809022089e-01
rank-1 residual        : 2.408324946023e-01


In [ ]:
# ============================================================
# Cell 11 — ORACLE CHANGE-SUPPORT Step 3
# ============================================================
# Validation-only branch: give Step 3 the true 4 changed edges at each
# transition, but still do NOT give it the interaction law or edge weights.
# This isolates dynamics reconstruction from Step-2 support errors.
# ============================================================
TRUE_SUPPORT_ARRAYS = tuple(
    np.asarray([EDGE_INDEX_1B[e] for e in support], dtype=int)
    for support in transition_supports_true
)

STEP3_ORACLE_SUPPORT = reconstruct_vector_field_from_observations(
    Y=V_MID,
    D=D,
    edge_features=EDGE_PSI,
    stage_of_sample=OBS_STAGE,
    change_constraints_or_supports=TRUE_SUPPORT_ARRAYS,
    solver_method="dense_lstsq",
    solver_kwargs={
        "rcond": None,
        "compute_raw_svd_diagnostics": True,
        "verbose": True,
    },
    design_mode="dense",
    return_design=True,
)

print("=" * 90)
print("ORACLE-SUPPORT STEP 3")
print("=" * 90)
print("parameters              :", STEP3_ORACLE_SUPPORT.parameter_count)
print("rank                    :", f"{STEP3_ORACLE_SUPPORT.rank}/{STEP3_ORACLE_SUPPORT.parameter_count}")
print("identifiable            :", STEP3_ORACLE_SUPPORT.identifiable)
print("relative residual       :", f"{STEP3_ORACLE_SUPPORT.relative_residual:.12e}")
print("uncertainty floor       :", f"{UNCERTAINTY_FLOOR:.12e}")
print("oracle info used        : TRUE CHANGE SUPPORT ONLY")


In [ ]:
# ============================================================
# Cell 12 — Current Step 4 on oracle-support reconstruction
# ============================================================
STEP4_ORACLE_SUPPORT = decompose_vector_field_sources(
    STEP3_ORACLE_SUPPORT,
    hypothesis="shared_interaction_law",
    normalization="unit_norm",
    reference_component=0,
    component_labels=LOCAL_LIBRARY.component_labels,
)

print("=" * 90)
print("STEP 4 — ORACLE CHANGE SUPPORT")
print("=" * 90)
THETA_ORACLE_UNIT = np.asarray(STEP4_ORACLE_SUPPORT.theta, dtype=float)
if abs(THETA_ORACLE_UNIT[0]) > 1e-10 * np.max(np.abs(THETA_ORACLE_UNIT)):
    THETA_ORACLE_GAUGE = THETA_ORACLE_UNIT / THETA_ORACLE_UNIT[0]
else:
    THETA_ORACLE_GAUGE = None

print("theta unit             :", np.array2string(THETA_ORACLE_UNIT, precision=12))
print("theta reference gauge  :", None if THETA_ORACLE_GAUGE is None else np.array2string(THETA_ORACLE_GAUGE, precision=12))
print("Taylor reference       :", np.array2string(THETA_TAYLOR_GAUGE, precision=12))
print("rank-1 energy fraction :", f"{STEP4_ORACLE_SUPPORT.rank1_energy_fraction:.12e}")
print("s2/s1                  :", f"{STEP4_ORACLE_SUPPORT.second_to_first_singular_ratio:.12e}")
print("rank-1 residual        :", f"{STEP4_ORACLE_SUPPORT.relative_residual:.12e}")

if THETA_ORACLE_GAUGE is not None:
    poly_theta_rel_to_taylor = (
        np.linalg.norm(THETA_ORACLE_GAUGE - THETA_TAYLOR_GAUGE)
        / np.linalg.norm(THETA_TAYLOR_GAUGE)
    )
    print("theta vs Taylor rel.err:", f"{poly_theta_rel_to_taylor:.12e}")
else:
    poly_theta_rel_to_taylor = np.inf
    print("theta vs Taylor rel.err: unavailable (linear component unstable)")


In [ ]:
# ============================================================
# Cell 13 — ORACLE microscopic-W representation control
# ============================================================
# Validation only.  Here the true active topology AND edge weights are fixed,
# so the only unknown is the shared polynomial interaction law.  This is a
# linear regression in theta and tests whether the polynomial representation
# itself can recover the Kuramoto sine law at the current data resolution.
#
# This is NOT the intended inference algorithm.  Recovering W and theta jointly
# from the observational equivalence class is the Step-4 optimization problem.
# ============================================================

# Build node-space design for one shared theta using the true W schedule.
tail_idx = np.array([i - 1 for i, _ in CANDIDATE_EDGES], dtype=int)
head_idx = np.array([j - 1 for _, j in CANDIDATE_EDGES], dtype=int)

A_THETA_ORACLE_W = np.zeros((len(V_MID) * N, L), dtype=float)
for q in range(len(V_MID)):
    r = int(OBS_STAGE[q])
    for edge in SNAPSHOTS[r]:
        m = EDGE_INDEX_1B[edge]
        w = EDGE_WEIGHT[edge]
        A_THETA_ORACLE_W[q*N + tail_idx[m], :] += w * EDGE_PSI[q, m, :, 0]
        A_THETA_ORACLE_W[q*N + head_idx[m], :] += w * EDGE_PSI[q, m, :, 1]

y_theta = V_MID.ravel()
col_scale = np.linalg.norm(A_THETA_ORACLE_W, axis=0)
good = col_scale > 1e-14
A_scaled = A_THETA_ORACLE_W[:, good] / col_scale[good][None, :]
coef_scaled, _, rank_theta, s_theta = np.linalg.lstsq(A_scaled, y_theta, rcond=None)
THETA_ORACLE_W_RAW = np.zeros(L, dtype=float)
THETA_ORACLE_W_RAW[good] = coef_scaled / col_scale[good]

pred_theta = A_THETA_ORACLE_W @ THETA_ORACLE_W_RAW
rho_theta_oracle_w = np.linalg.norm(pred_theta - y_theta) / np.linalg.norm(y_theta)

# Exact Taylor coefficients in the normalized z=d/s coordinate BEFORE gauge fixing:
# sin(d)=sin(s z)=sum_p c_p z^p.
THETA_TAYLOR_RAW = np.zeros(L, dtype=float)
for p in range(1, L + 1):
    if p % 2 == 1:
        k = (p - 1) // 2
        THETA_TAYLOR_RAW[p - 1] = ((-1.0)**k) * DIFF_SCALE**p / math.factorial(p)

raw_theta_rel_to_taylor = (
    np.linalg.norm(THETA_ORACLE_W_RAW - THETA_TAYLOR_RAW)
    / np.linalg.norm(THETA_TAYLOR_RAW)
)

print("=" * 90)
print("ORACLE-W POLYNOMIAL LAW RECOVERY")
print("=" * 90)
print("design shape               :", A_THETA_ORACLE_W.shape)
print("rank                       :", f"{rank_theta}/{L}")
print("relative residual          :", f"{rho_theta_oracle_w:.12e}")
print("uncertainty floor          :", f"{UNCERTAINTY_FLOOR:.12e}")
print("inferred raw theta         :", np.array2string(THETA_ORACLE_W_RAW, precision=12))
print("Taylor raw theta           :", np.array2string(THETA_TAYLOR_RAW, precision=12))
print("theta vs Taylor rel.err    :", f"{raw_theta_rel_to_taylor:.12e}")
print("oracle information used    : TRUE ACTIVE TOPOLOGY + TRUE EDGE WEIGHTS")


In [ ]:
# ============================================================
# Cell 14 — Polynomial -> analytic sine compression
# ============================================================
# Use the oracle-W polynomial control above to test the symbolic-compression
# question in isolation.  THETA_ORACLE_W_RAW represents
#
#   f_poly(d) = sum_p theta_p (d/s)^p.
#
# For the true Kuramoto law this should approximate sin(d).  We then fit
# A*sin(omega*d) WITHOUT supplying A=omega=1.
# ============================================================
from scipy.optimize import least_squares

THETA_HAT = np.asarray(THETA_ORACLE_W_RAW, dtype=float)

# Active-edge observed relative-coordinate samples only.
d_active = []
for q in range(len(X_MID)):
    r = int(OBS_STAGE[q])
    xq = X_MID[q]
    for edge in SNAPSHOTS[r]:
        i, j = edge
        d_active.append(xq[j - 1] - xq[i - 1])
d_active = np.asarray(d_active, dtype=float)


def physical_polynomial_law(d, theta):
    z = np.asarray(d, dtype=float) / DIFF_SCALE
    out = np.zeros_like(z, dtype=float)
    for p, coeff in enumerate(theta, start=1):
        out += coeff * z**p
    return out

f_poly_active = physical_polynomial_law(d_active, THETA_HAT)
f_true_active = np.sin(d_active)
poly_vs_true_rel = (
    np.linalg.norm(f_poly_active - f_true_active)
    / np.linalg.norm(f_true_active)
)


def residual_sine(params):
    A, omega = params
    return A * np.sin(omega * d_active) - f_poly_active

fit = least_squares(
    residual_sine,
    x0=np.array([1.0, 1.0]),
    bounds=([0.0, 0.0], [10.0, 10.0]),
    xtol=1e-14,
    ftol=1e-14,
    gtol=1e-14,
    max_nfev=20000,
)
A_hat, omega_hat = fit.x
f_sine_hat = A_hat * np.sin(omega_hat * d_active)
sine_compression_rel = (
    np.linalg.norm(f_sine_hat - f_poly_active)
    / max(np.linalg.norm(f_poly_active), np.finfo(float).tiny)
)
sine_vs_true_rel = (
    np.linalg.norm(f_sine_hat - f_true_active)
    / np.linalg.norm(f_true_active)
)

# Convert raw z-basis coefficients into physical monomial coefficients
# c_p in sum_p c_p d^p.
physical_coeff = np.array([
    THETA_HAT[p - 1] * DIFF_SCALE**(-p)
    for p in range(1, L + 1)
])

omega2_from_c3 = np.nan
omega2_from_c5 = np.nan
if L >= 3 and abs(physical_coeff[0]) > 1e-14:
    omega2_from_c3 = -6.0 * physical_coeff[2] / physical_coeff[0]
if L >= 5 and abs(physical_coeff[2]) > 1e-14:
    omega2_from_c5 = -20.0 * physical_coeff[4] / physical_coeff[2]

print("=" * 90)
print("POLYNOMIAL -> SINE COMPRESSION")
print("=" * 90)
print("sampled active d range      :", (float(d_active.min()), float(d_active.max())))
print("physical polynomial coeffs  :", np.array2string(physical_coeff, precision=12))
print("poly vs true sin rel.err    :", f"{poly_vs_true_rel:.12e}")
print("fitted A                    :", f"{A_hat:.12e}")
print("fitted omega                :", f"{omega_hat:.12e}")
print("A*omega                     :", f"{A_hat * omega_hat:.12e}")
print("sine compression rel.err    :", f"{sine_compression_rel:.12e}")
print("fitted sine vs true rel.err :", f"{sine_vs_true_rel:.12e}")
print("omega^2 from c3/c1          :", omega2_from_c3)
print("omega^2 from c5/c3          :", omega2_from_c5)
print("true analytic target        : A=1, omega=1")

# Visual diagnostic over the active observed domain.
d_grid = np.linspace(d_active.min(), d_active.max(), 600)
plt.figure(figsize=(8, 5))
plt.plot(d_grid, np.sin(d_grid), label="true sin(d)")
plt.plot(d_grid, physical_polynomial_law(d_grid, THETA_HAT), '--', label="recovered polynomial")
plt.plot(d_grid, A_hat * np.sin(omega_hat * d_grid), ':', label="compressed A sin(omega d)")
plt.xlabel("d = x_neighbor - x_self")
plt.ylabel("interaction law")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 15 — Optional direct Taylor-resolution audit
# ============================================================
# This is oracle-only and is here to explain why degree 11 was chosen for
# this numerical-resolution benchmark.  The inference itself never receives
# a sine candidate.
import math

print("degree    relative approximation error on sampled d")
print("-" * 60)
for deg in (1, 3, 5, 7, 9, 11, 13):
    approx = np.zeros_like(d_all)
    for k in range((deg + 1) // 2):
        p = 2*k + 1
        approx += ((-1.0)**k) * d_all**p / math.factorial(p)
    rel = np.linalg.norm(approx - np.sin(d_all)) / np.linalg.norm(np.sin(d_all))
    print(f"{deg:>6d}    {rel:.12e}")
print("uncertainty floor:", f"{UNCERTAINTY_FLOOR:.12e}")


In [ ]:
# ============================================================
# Cell 16 — Summary
# ============================================================
print("=" * 90)
print("N=8 KURAMOTO / POLYNOMIAL-DISCOVERY SUMMARY")
print("=" * 90)
print("Step 1 exact transitions        :", blind_exact)
print("Polynomial degree               :", POLY_DEGREE)
print("Relative-coordinate scale       :", f"{DIFF_SCALE:.6e}")
print("Taylor truncation rel.err       :", f"{taylor_rel_error:.6e}")
print("Step 2 selected groups          :", len(scalable_set))
print("Step 2 TP / FP / FN             :", TP, FP, FN)
print("Step 2 recall                   :", f"{recall:.3f}")
print("Blind Step 3 residual           :", f"{STEP3_BLIND.relative_residual:.6e}")
print("Blind Step 4 rank-1 residual    :", f"{STEP4_BLIND.relative_residual:.6e}")
print("Oracle-support Step 3 residual  :", f"{STEP3_ORACLE_SUPPORT.relative_residual:.6e}")
print("Oracle-support Step 4 rank1 res :", f"{STEP4_ORACLE_SUPPORT.relative_residual:.6e}")
print("Oracle-W theta/Taylor rel.err   :", f"{raw_theta_rel_to_taylor:.6e}")
print("Oracle-W polynomial residual    :", f"{rho_theta_oracle_w:.6e}")
print("poly -> true sin rel.err        :", f"{poly_vs_true_rel:.6e}")
print("compressed sine A               :", f"{A_hat:.8f}")
print("compressed sine omega           :", f"{omega_hat:.8f}")
print("sine compression rel.err        :", f"{sine_compression_rel:.6e}")
print("\nInterpretation: support selection, current-Step4 representative dependence, and polynomial->sine representability are reported separately.")
